# Supervised Fine-Tuning (SFT) with Serverless Customization on SageMaker AI

## Lab 1 — Contract Review That Cites Its Evidence: Prepare the Data

This is the first of four interconnected labs:

| Lab | Notebook | What you'll do |
|-----|----------|----------------|
| **Lab 1** | 1-prepare-data.ipynb ← *you are here* | Download ContractNLI, build training records, and register datasets |
| **Lab 2** | 2-fine-tune-llm.ipynb | Submit a serverless LoRA fine-tuning job and register the result |
| **Lab 3** | 3-evaluation.ipynb | Score the base, frontier, and fine-tuned model on held-out contracts |
| **Lab 4** | 4-deployment.ipynb | Deploy the merged model to a real-time SageMaker endpoint |

### What you'll do in this lab

1. **Download the dataset** — [ContractNLI](https://stanfordnlp.github.io/contract-nli/) (CC-BY-4.0), 607 real NDAs annotated against a fixed legal checklist
2. **Build training records** — one prompt/completion pair per contract, with the prompt identical to what the model sees at inference time
3. **Upload** the three splits (train / val / test) to Amazon S3
4. **Register** them as versioned DataSet assets in the SageMaker AI Registry

---


### The Task

[ContractNLI](https://stanfordnlp.github.io/contract-nli/) is a **document-level Natural Language Inference (NLI)** task for automated contract review. Given an NDA and a fixed set of 17 hypotheses, the model must:

1. **Classify** each hypothesis as Entailment, Contradiction, or NotMentioned relative to the contract
2. **Identify evidence** — the specific numbered spans (clauses) that justify each Entailment or Contradiction decision

<img src="images/hypothesis_example.png" width="720">

Each hypothesis in the checklist is a statement about what a contract *should* say. For each one, the model reads the full contract and decides:

- **Entailment** — the contract explicitly states or implies the hypothesis is true. A clause directly supports it, and the model must cite that clause number.
- **Contradiction** — the contract says something that *conflicts* with the hypothesis. This is subtler than just "false" — it means the contract actively says the opposite. A broad prohibition with no exception can contradict a hypothesis even if the hypothesis topic isn't mentioned by name.
- **NotMentioned** — the contract simply doesn't address the hypothesis at all. No clause supports it, no clause conflicts with it.

The distinction between Contradiction and NotMentioned is what makes this hard. When a contract is silent on something it's NotMentioned, but when it contains a broad prohibition that leaves no room for the hypothesis to be true, that's a Contradiction. The model has to read carefully to tell the difference.

In the dataset, each contract/hypothesis pair is annotated with:

```json
{
  "choice": "Entailment | Contradiction | NotMentioned",
  "spans": [3, 4]  // clause indices that justify the decision; empty for NotMentioned
}
```

A model does this for all 17 hypotheses at once, in a single reply. That reply is what the next section shows.

### The Expected Output

The model responds with strict JSON — one entry per hypothesis, each with a label and evidence spans. Each key such as nda-11 identifies one of the 17 checklist hypotheses, not the contract — the same 17 keys appear in the output for every contract, since every contract is checked against the same checklist. The keys run from 1 to 20 rather than 1 to 17: three numbers (6, 9, and 14) don't exist in ContractNLI's original numbering, so 17 hypotheses land on IDs that reach 20.

```json
{
  "nda-11": {"label": "NotMentioned",  "evidence": []},
  "nda-16": {"label": "NotMentioned",  "evidence": []},
  "nda-15": {"label": "NotMentioned",  "evidence": []},
  "nda-10": {"label": "NotMentioned",  "evidence": []},
  "nda-2":  {"label": "Contradiction", "evidence": [3, 4]},
  "nda-1":  {"label": "NotMentioned",  "evidence": []},
  "nda-19": {"label": "NotMentioned",  "evidence": []},
  "nda-12": {"label": "NotMentioned",  "evidence": []},
  "nda-20": {"label": "NotMentioned",  "evidence": []},
  "nda-3":  {"label": "NotMentioned",  "evidence": []},
  "nda-18": {"label": "NotMentioned",  "evidence": []},
  "nda-7":  {"label": "Contradiction", "evidence": [3]},
  "nda-17": {"label": "NotMentioned",  "evidence": []},
  "nda-8":  {"label": "NotMentioned",  "evidence": []},
  "nda-13": {"label": "NotMentioned",  "evidence": []},
  "nda-5":  {"label": "Contradiction", "evidence": [3]},
  "nda-4":  {"label": "Entailment",    "evidence": [4]}
}
```

Producing this JSON for a held-out contract, reliably, is the whole goal of fine-tuning. The rest of this lab builds the training data that teaches a model to do it.

### The Data

The training data for this task is [ContractNLI](https://stanfordnlp.github.io/contract-nli/) (Koreeda & Manning, *Findings of EMNLP 2021*) — 607 real NDAs sourced from SEC filings and the public web, each annotated against the same 17 hypotheses shown above. Released under **CC-BY-4.0**.

| Split | Contracts | Used for |
|-------|-----------|---------|
| Train | 423 | Fine-tuning |
| Dev | 61 | Validation during training |
| Test | 123 | Held-out evaluation in Lab 3 |

Splits are document-level — no contract appears in more than one split. The rest of this notebook turns these 607 contracts into prompt/completion records shaped exactly like the JSON above, then uploads and registers them so Lab 2 can train on them.

---

### Install requirements

In [ ]:
%pip install -r requirements.txt

### Set up the SageMaker session

We start by creating a **SageMaker Session** — a lightweight helper that manages the connection to your AWS account. It:
- Resolves the **default S3 bucket** for staging datasets and model artifacts
- Reads your **IAM execution role** — the identity that grants SageMaker permission to read S3, write metrics, and launch training jobs on your behalf

> **Tip:** If you're running inside SageMaker Studio, get_execution_role() automatically retrieves the Studio execution role. Outside Studio, you can create a role named sagemaker_execution_role in IAM with the AmazonSageMakerFullAccess managed policy attached.


In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

### contractnli.py

Every notebook starts with import contractnli as C, a helper module for downloading data, reading it, and rendering the prompt. These are the helper functions:

| Call | Returns | Used for |
|---|---|---|
| C.ensure_dataset("./data") | the unpacked path | downloads the ContractNLI archive once |
| C.load(split) | documents, checklist | reads train, dev or test |
| C.doc_spans(doc) | numbered clause list | one contract as numbered clauses |
| C.gold_for(doc) | expert answer dict | the expert answer for one contract |
| C.build_prompt(doc, labels) | one string | the whole request: instruction, contract, checklist |

One constant matters too: C.INSTRUCTION, the template build_prompt fills in — you'll print it below.

### Download the dataset

The checklist is identical across all three splits, so only the train copy is kept as labels below, and that one dict renders the checklist into every prompt.

In [ ]:
import contractnli as C

C.ensure_dataset("./data")

train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")

print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

#### What one document actually looks like

Before building anything, it is worth seeing the raw shape you are working from. The
dataset gives you documents; the three helpers below are how you get from a document to
the pieces the prompt needs.


In [ ]:
doc_example = train_docs[0]

print("One ContractNLI document is a plain dict. Its fields:\n")
for key, value in doc_example.items():
    size = f"{len(value):,} items" if isinstance(value, list) else f"{len(str(value)):,} chars"
    print(f"  doc[{key!r}]:{' ' * (20 - len(key))}{type(value).__name__:5s} {size}")

print("\nOnly three of those matter here, and `contractnli.py` has a helper for each.\n")

# 1. The contract, cut into the clauses the model will cite by number.
print("1. doc['spans'] holds (start, end) offsets into doc['text'] — the dataset's own")
print("   clause split. C.doc_spans(doc) slices them out and numbers them:\n")
for number, text in C.doc_spans(doc_example)[:3]:
    print(f"     [{number}] {text[:62]}")

# 2. The expert answer. `choice` is the verdict, `spans` the clauses that justify it.
print("\n2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it")
print("   to one entry per checklist item:\n")
gold_example = C.gold_for(doc_example)
for key in list(labels)[:2]:
    print(f"     {key}: {gold_example[key]}")

# 3. The checklist is the same for every contract. In this format it is rendered into
#    every prompt, which is why it comes from one dict rather than per-record text.
print("\n3. `labels`, the second value C.load() returned, is the checklist itself:\n")
first_key = list(labels)[0]
print(f"     labels[{first_key!r}]:")
for field, text in labels[first_key].items():
    print(f"       {field}: {text[:66]}")

### The checklist the model has to answer

The 17 hypotheses, each with the short description that names it. build_prompt() renders these into the checklist block of the prompt, so every contract is judged against this same list.

The sort key below is for readability only: the dataset's numbering starts at nda-11, and plain sorting would put nda-10 before nda-2. Numbers 6, 9 and 14 are unused, which is why 17 items reach nda-20.

In [ ]:
for k, v in sorted(labels.items(), key=lambda kv: int(kv[0].split("-")[1])):
    print(f"{k:7s} [{v['short_description']}]")
    print(f"        {v['hypothesis']}")

### Look at one real contract

Below is one of the shortest NDAs in the test set, split into numbered spans, followed by the gold answer. Read span [3], then look at nda-5 (sharing with employees) and nda-7 (sharing with third parties).

In [ ]:
doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]
spans = C.doc_spans(doc)

print(f"{doc['file_name']}  —  {len(doc['text'].split())} words, {len(spans)} spans\n")
for i, t in spans:
    print(f"[{i}] {t[:160]}")

The gold answer for the same contract, one line per checklist item: a choice from the three labels, and the span numbers the annotator pointed to, in the same numbering as the markers above. Spans are non-empty only for Entailment and Contradiction entries.

Further down, in "Build the training records," gold_json() converts this same gold answer into the JSON string the model is trained to produce — the completion shown in "The Expected Output" above — by renaming choice to label and spans to evidence.

In [ ]:
gold = C.gold_for(doc)
print("gold answer:\n")
for k in sorted(gold, key=lambda x: int(x.split("-")[1])):
    v = gold[k]
    ev = f"  evidence={v['spans']}" if v["spans"] else ""
    print(f"{k:7s} {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

### Check the answer against the clause you just read

This is the contract from the introduction, and span [3] is that blanket prohibition. It drives three separate Contradiction verdicts:

| Item | Subject | Evidence |
|---|---|---|
| nda-5 | sharing with employees | [3] |
| nda-7 | sharing with third parties | [3] |
| nda-2 | only technical information is confidential | [3, 4] |

One clause, three verdicts — the model has to reason over the whole document per item, not retrieve one passage per question. 13 of the 17 items here are NotMentioned. Short NDAs are silent on most of the checklist.

### The prompt

Everything the model sees is one string: instruction, contract as numbered spans, then the checklist. The checklist comes last, so it's what the model reads right before answering. It's defined once, in contractnli.py, and used by every notebook, so the training prompt and the inference prompt can never drift apart.

In [ ]:
# The exact template. {n}, {spans} and {checklist} are the only substitutions. The
# prompt the model receives is this plus one more line: build_prompt() appends
# C.NO_THINK at the end, where the switch has to be.
print("=" * 70, "\nINSTRUCTION\n", "=" * 70, sep="")
print(C.INSTRUCTION)
print(f"\n{C.NO_THINK}    <- appended by build_prompt(doc, labels); no_think=False omits it")


To experiment with the wording, set C.INSTRUCTION here and re-run the record build below — every notebook then picks up your version:

```python
C.INSTRUCTION = """...your wording, keeping {n}, {spans} and {checklist}..."""
```

The assertions in the next cell check that every stored prompt still matches build_prompt() exactly.

### Build the training records

Each record is a prompt/completion pair (query/response for test). Only the completion is supervised, so the model learns to produce the JSON rather than reproduce the contract. The recipe caps records at 4096 tokens and drops anything longer, so only 315 of 423 contracts train by default — raise dataset_max_len in notebook 2 to train on all of them. Below: the same work on one contract, step by step, before building all 607 records at once.

In [ ]:
import json

label_keys = list(labels.keys())          # the checklist's own order, reused below


def gold_json(doc):
    """The expert answer for one contract, as the exact JSON the model must emit."""
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})


# Step 1 — the target. gold_json only renames fields; the judgement is the annotator's.
item = label_keys[1]
print("the dataset stores:      ", json.dumps(C.gold_for(doc_example)[item]))
print("the prompt asks for:     ", json.dumps(json.loads(gold_json(doc_example))[item]))
print("                          ^ choice -> label, spans -> evidence, for all 17 items")

# Step 2 — the two halves of the record, and what each is made of.
prompt = C.build_prompt(doc_example, labels)
target = gold_json(doc_example)
contract_chars = len("\n".join(f"[{i}] {t}" for i, t in C.doc_spans(doc_example)))

print(f"\nC.build_prompt(doc, labels) {len(prompt):6,d} chars   the whole request")
print(f"  of which the contract      {contract_chars:6,d} chars   the only part that varies")
print(f"  fixed instruction etc.     {len(prompt) - contract_chars:6,d} chars   identical every time")
print(f"gold_json(doc)               {len(target):6,d} chars   what the model must produce")

# Step 3 — the record is those two strings under two keys. That is the whole format.
record = {"prompt": prompt, "completion": target}
print(f"\nrecord keys: {list(record)}")
print("\nOne function, two uses: the same build_prompt() output is the `prompt` of a")
print("training record here and the request sent at inference time in notebooks 3 and 4.")
print("That is why they cannot drift apart.")


Now all three splits at once. Three things happen, in order.

1. **make_records and make_test_records** map documents to records. The first produces the prompt/completion pair for train and val; the second produces genqa's query/response for test. Both call the same build_prompt(), so the test split is a renaming of the same string, not a second prompt.
2. **The print loop** averages characters per record, as a rough stand-in for a tokenizer.
3. **The assertions** are the no-skew promise made executable: every stored prompt must equal build_prompt() for its document, and every completion must parse as non-empty JSON.

If an assertion trips, the message names the split and the cause — that's the point of having them here rather than discovering the mismatch during training.

In [ ]:
# `label_keys` and `gold_json` come from the cell above. The two builders below are
# the only new code: each maps a list of documents to a list of records.

def make_records(docs):
    """Training records: one prompt/completion pair per contract."""
    return [{"prompt": C.build_prompt(d, labels), "completion": gold_json(d)}
            for d in docs]


def make_test_records(docs):
    """Evaluation records. The managed scorer reads `query`/`response` (genqa) and
    nothing else, so the test split keeps that shape — see the note below."""
    return [{"query": C.build_prompt(d, labels), "response": gold_json(d)}
            for d in docs]


records = {"train": make_records(train_docs),
           "val": make_records(dev_docs),
           "test": make_test_records(test_docs)}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)
    print(f"{name:5s}: {len(rows):4d} records, avg prompt {avg:6d} chars "
          f"(~{avg // 4} tokens)")

# The byte-identical promise, enforced rather than asserted in prose: every stored
# prompt must be exactly what build_prompt() produces at inference time, and the
# completion must be the only supervised target.
for split, docs in (("train", train_docs), ("val", dev_docs), ("test", test_docs)):
    prompt_field = "query" if split == "test" else "prompt"
    target_field = "response" if split == "test" else "completion"
    for rec, doc in zip(records[split], docs):
        assert set(rec) == {prompt_field, target_field}, f"unexpected fields in {split}"
        assert rec[prompt_field] == C.build_prompt(doc, labels), (
            f"prompt drift in {split}: stored record differs from build_prompt()")
        assert json.loads(rec[target_field]), f"empty target in {split}"

total = sum(len(rows) for rows in records.values())
print(f"verified: all {total} stored prompts are byte-identical to build_prompt() output")


One complete training record — the prompt the model reads and the JSON it must produce. The cell picks the train record with the shortest prompt, so one whole contract fits on screen. It's the smallest NDA in the split, not a typical one.

In [ ]:
sample = min(records["train"], key=lambda r: len(r["prompt"]))

print("=" * 70, "\nPROMPT\n", "=" * 70, sep="")
print(sample["prompt"])
print("\n" + "=" * 70, "\nCOMPLETION\n", "=" * 70, sep="")
print(json.dumps(json.loads(sample["completion"]), indent=1)[:800], "...")


#### Write to disk and upload to Amazon S3

shutil.rmtree removes ./sft_data first, so a re-run can't leave a stale file behind. Each split is written as one JSON object per line, and the dev documents are written as val — that's the name notebook 2 fetches the validation set by. Each file then uploads to S3, and the three URIs accumulate in s3_paths for the next cell to register.

In [ ]:
import pathlib
import shutil

from config import DATASET_PREFIX

local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)

for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")

s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), bucket_name, key)
    s3_paths[name] = f"s3://{bucket_name}/{key}"
    print(s3_paths[name])

#### Register the datasets

DataSet.create writes a registry entry pointing at the S3 object you just uploaded. Notebook 2 and 3 fetch these by name instead of an S3 URI:

| Dataset | Technique | Consumed by |
|---|---|---|
| contractnli-nda-review-train | SFT | notebook 2, training_dataset |
| contractnli-nda-review-val | SFT | notebook 2, validation_dataset |
| contractnli-nda-review-test | none | notebook 3, dataset |


In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


def register(name, source, technique=None):
    kwargs = dict(name=name, source=source, wait=True)
    if technique is not None:
        kwargs["customization_technique"] = technique
    ds = DataSet.create(**kwargs)
    print(f"created dataset: {name}")
    return ds


training_dataset = register(f"{DATASET_PREFIX}-train", s3_paths["train"], CustomizationTechnique.SFT)
val_dataset = register(f"{DATASET_PREFIX}-val", s3_paths["val"], CustomizationTechnique.SFT)
test_dataset = register(f"{DATASET_PREFIX}-test", s3_paths["test"])

### What you built

contractnli-nda-review-train (423 records), -val (61) and -test (123 held out), registered and ready. Notebooks 2 and 3 look them up by name.

Continue to notebook 2 to run the serverless LoRA fine-tuning job.